# 03 — Modelagem

**Dimensão 5 da rúbrica — 20 pontos.**

Mínimo de **dois** classificadores distintos. Um único modelo zera 8 dos 20 pontos.

In [10]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.svm import SVC

RANDOM_STATE = 42

RAW = Path("..") / "data" / "raw" / "df.csv"
PROCESSED = Path("..") / "data" / "processed"
TARGET = "STATUS"

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv(PROCESSED / "dataset_tratado.csv")
df.head()

,FLAG_OWN_CAR,FLAG_OWN_REALTY,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,STATUS,INCOME_PER_MEMBER,AMT_INCOME_TOTAL,AGE,YEARS_EMPLOYED,CNT_CHILDREN,CNT_FAM_MEMBERS,hot_F,hot_M,hot_Commercial associate,hot_Pensioner,hot_State servant,hot_Student,hot_Working,hot_Academic degree,hot_Higher education,hot_Incomplete higher,hot_Lower secondary,hot_Secondary / secondary special,hot_Civil marriage,hot_Married,hot_Separated,hot_Single / not married,hot_Widow,hot_Co-op apartment,hot_House / apartment,hot_Municipal apartment,hot_Office apartment,hot_Rented apartment,hot_With parents,hot_Accountants,hot_Cleaning staff,hot_Cooking staff,hot_Core staff,hot_Drivers,hot_HR staff,hot_High skill tech staff,hot_IT staff,hot_Laborers,hot_Low-skill Laborers,hot_Managers,hot_Medicine staff,hot_Private service staff,hot_Realty agents,hot_Sales staff,hot_Secretaries,hot_Security staff,hot_Unemployed,hot_Unknown,hot_Waiters/barmen staff
0,1,1,1,0,0,0,6.482856,2.006494,-0.956283,1.059145,-0.558271,-0.201770,False,True,False,False,False,False,True,False,True,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
1,1,1,0,0,0,0,5.815359,-0.749305,1.302080,-0.372449,-0.558271,-0.201770,False,True,False,False,False,False,True,False,False,False,False,True,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False
2,0,1,0,1,1,0,12.506181,1.057895,0.780919,0.422881,-0.558271,-1.266753,True,False,True,False,False,False,False,False,False,False,False,True,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False
3,0,1,0,0,0,0,12.554971,1.158611,1.562660,-0.849647,-0.558271,-1.266753,True,False,False,True,False,False,False,False,True,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False
4,1,1,1,1,1,0,6.253090,1.057895,0.259758,-0.531515,-0.558271,-0.201770,False,True,False,False,False,False,True,False,True,False,False,False,False,True,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


## 1. Split treino/teste

`stratify` preserva a proporção das classes nos dois conjuntos.

In [11]:
# Variáveis independentes
X = df.drop(columns=[TARGET])

# Variável Alvo
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

## 2. Modelos candidatos

Usar `Pipeline` evita vazamento: o scaler é ajustado só no fold de treino durante a validação cruzada.

In [12]:



knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)

logreg = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
logreg.fit(X_train, y_train)
y_pred_logreg = logreg.predict(X_test)

tree = DecisionTreeClassifier(random_state=RANDOM_STATE, max_depth=10)
tree.fit(X_train, y_train)
y_pred_tree = tree.predict(X_test)

forest = RandomForestClassifier(random_state=RANDOM_STATE, max_depth=10, n_estimators=100)
forest.fit(X_train, y_train)
y_pred_forest = forest.predict(X_test)

svm = SVC(random_state=RANDOM_STATE)
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)


In [4]:
modelos = {
    "KNN": Pipeline([
        ("clf", KNeighborsClassifier(n_neighbors=5, n_jobs=-1)),
    ]),
    "Regressão Logística": Pipeline([
        ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced')),
    ]),
    "Floresta Aleatória": Pipeline([
        ("clf", RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced')),
    ]),
    "Árvore de Decisão": Pipeline([
        ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced')),
    ]),
    "Hist Gradient Boosting": Pipeline([
        ("clf", HistGradientBoostingClassifier(class_weight='balanced', random_state=RANDOM_STATE))
    ])
}

## 3. Validação cruzada

In [5]:
from sklearn.model_selection import cross_val_score

for nome, modelo in modelos.items():
    f1 = cross_val_score(modelo, X_train, y_train, cv=5, scoring="f1")
    acc = cross_val_score(modelo, X_train, y_train, cv=5, scoring="accuracy")
    print(f'F1: {f1.mean():.4f}\t Accuracy: {acc.mean():.4f}\t{nome}')

F1: 0.0056	 Accuracy: 0.9543	KNN
F1: 0.0969	 Accuracy: 0.6197	Regressão Logística
F1: 0.0728	 Accuracy: 0.9318	Floresta Aleatória
F1: 0.0695	 Accuracy: 0.8975	Árvore de Decisão
F1: 0.1137	 Accuracy: 0.8732	Hist Gradient Boosting


In [6]:
from sklearn.metrics import average_precision_score, roc_auc_score

baseline_results = []

for nome, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    y_proba = modelo.predict_proba(X_test)[:, 1]
    
    roc_auc = roc_auc_score(y_test, y_proba)
    pr_auc = average_precision_score(y_test, y_proba)
    
    baseline_results.append({
        "Modelo": nome,
        "ROC-AUC": round(roc_auc, 4),
        "PR-AUC": round(pr_auc, 4)
    })

df_resultados = pd.DataFrame(baseline_results).sort_values(by="ROC-AUC", ascending=False)
print(df_resultados.to_string(index=False))

                Modelo  ROC-AUC  PR-AUC
Hist Gradient Boosting   0.5826  0.0819
   Regressão Logística   0.5598  0.1085
                   KNN   0.5439  0.0731
     Árvore de Decisão   0.5086  0.0461
    Floresta Aleatória   0.4759  0.0534


In [7]:
from sklearn.model_selection import StratifiedKFold, cross_validate


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ['average_precision', 'roc_auc', 'f1', 'recall']

resultados = []

for nome, modelo in modelos.items():
    scores = cross_validate(modelo, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    resultados.append({
        "Modelo": nome,
        "PR-AUC (Média)": scores['test_average_precision'].mean(),
        "ROC-AUC (Média)": scores['test_roc_auc'].mean(),
        "F1-Score (Média)": scores['test_f1'].mean(),
        "Recall (Média)": scores['test_recall'].mean()
    })

df_ranking = pd.DataFrame(resultados).sort_values(by="PR-AUC (Média)", ascending=False)
print(df_ranking.to_string(index=False))

                Modelo  PR-AUC (Média)  ROC-AUC (Média)  F1-Score (Média)  Recall (Média)
   Regressão Logística        0.097126         0.537800          0.096239        0.434193
Hist Gradient Boosting        0.075613         0.552555          0.107259        0.199106
                   KNN        0.061677         0.533869          0.034658        0.018080
    Floresta Aleatória        0.057161         0.509273          0.079999        0.067850
     Árvore de Decisão        0.048539         0.512417          0.073489        0.090398


## 4. Comparação

**Leitura:** _qual modelo venceu e por qual margem? A diferença é relevante ou está dentro do ruído?_